In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
print(assistant.rag('How do I run Ollama locally?'))

To run Ollama locally, follow these steps:

1. **Install Ollama**: Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in the terminal.

2. **Run Ollama**: Open a terminal and type `ollama run llama3`. This will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface where you can type questions.

3. **Test the Ollama local server**: Run the command `curl http://localhost:11434` to test the server. You should receive a response similar to `{"models": [...]}`.

4. **Install the Python client**: Install the Python client with `pip install ollama`.

5. **Run a minimal Python example**: Use the following Python code to test Ollama:
   ```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "

In [8]:
print(assistant.rag("How do I run Olama locally?"))

There is no information provided about running Olama locally. The given context does not contain any details or instructions on how to run Olama. If you need assistance with running Olama, please provide more context or information about Olama and its requirements.


In [9]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
)

response.output_text

"You'd like to join a course, however, I need a bit more information. Could you please provide more details about the course you're interested in, such as its name, subject, or platform it's being hosted on? That way, I can give you a more accurate answer about whether it's still possible to join."

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
    

In [13]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        'required': ["query"],
        'additionalProperties': False
    }
}

In [15]:
response  =openai_client.responses.create(
    model='llama-3.3-70b-versatile',
    input=messages,
    tools=[search_tool]
)

In [16]:
len(response.output)

2

In [17]:
call = response.output[0]

In [18]:
call

ResponseReasoningItem(id='resp_01kv584mt8ebesdhtbmqszjw46', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed')

In [19]:
import json

args = json.loads(call.arguments)
args

AttributeError: 'ResponseReasoningItem' object has no attribute 'arguments'